# Exploração inicial do dataset Wiki-RfA

## 1. Introdução

**O que é:** Uma *Request for Adminship* (RfA) é o processo onde um editor
da Wikipédia pede para virar administrador. O pedido é feito pelo próprio
candidato ou por outro membro, e qualquer editor pode votar a favor, de forma
neutra ou contra. O dataset reúne votos de RfAs desde 2003 até maio de 2013.

**Como vira um grafo:** Cada usuário é um nó. Cada voto é um arco dirigido de quem
vota (`SOURCE`) para o candidato (`TARGET`), e o voto (`VOTE`) é o peso do arco.
Como há votos contra, os pesos negativos vêm dos próprios dados.

**Onde encontrar:**
- Arquivo usado: [Wiki-RfA no Kaggle](https://www.kaggle.com/datasets/boneacrabonjac/wiki-rfa)


### O que representam SOURCE, TARGET, VOTE e RESULT?
| Coluna | O que representa |
| --- | --- |
| `SOURCE` | Usuário que vota, o eleitor. |
| `TARGET` | Usuário que recebe o voto, o candidato a administrador. |
| `VOTE` | Voto do eleitor: -1 (contra), 0 (neutro) ou 1 (a favor). |
| `RESULT` | Resultado da eleição: -1 (candidato rejeitado) ou 1 (aceito). |

Cada linha é um voto de um eleitor em um candidato numa eleição.

### O sentido dos arcos: eleitor → candidato.
Cada voto vira um arco dirigido de `SOURCE` para `TARGET` (eleitor → candidato).
O grafo é dirigido porque o voto tem direção: o fato de A votar em B não implica
que B votou em A. O peso do arco é o `VOTE`.

### O significado dos valores negativos, neutros e positivos.
| Valor | Significado |
| --- | --- |
| 1 | Voto a favor da candidatura. |
| 0 | Voto neutro. |
| -1 | Voto contra a candidatura. |

Os pesos negativos vêm dos próprios dados (votos contra).

### A ausência de data e identificador da eleição.
A versão original do dataset no SNAP traz também o ano e a data de cada voto e o
comentário do eleitor. O arquivo usado aqui tem só 4 colunas: não há data nem
identificador da eleição.

**Consequência:** cada eleição só pode ser reconhecida pelo candidato (`TARGET`).
A fonte informa que um usuário pode concorrer mais de uma vez, então um mesmo par
`SOURCE → TARGET` pode aparecer repetido por ter votado em eleições diferentes.
Sem data nem identificador, não dá para separar essas eleições nem ordenar os votos
no tempo.

### O tratamento dos dados ausentes.
- **O que encontrei:** 
- **Onde ocorrem:** 
- **Mecanismo de ausência:** 
- **Consequência para o grafo:** 
- **Tratamento escolhido:** 
- **Justificativa e limites:** 

In [ ]:
import kagglehub
from pathlib import Path

# [IA] Codex – adaptar o destino do download via kaggle para a pasta do notebook
path = kagglehub.dataset_download(
    "boneacrabonjac/wiki-rfa",
    output_dir=str(Path("raw").resolve())
)

print("Path to dataset files:", path)
#pasta .complete gerada pelo proprio kagglehub indicando sucesso no download

Path to dataset files: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\data\dataset_parte2\raw


In [82]:
import pandas as pd
df = pd.read_csv(
    Path(path) / "wikiRfA.csv"
, index_col=0)

df.head(10)

,SOURCE,TARGET,VOTE,RESULT
0,Steel1943,BDD,1,1
1,Cuchullain,BDD,1,1
2,INeverCry,BDD,1,1
3,Cncmaster,BDD,1,1
4,Miniapolis,BDD,1,1
5,Sven Manguard,BDD,1,1
6,Ramaksoud2000,BDD,1,1
7,RockMagnetist,BDD,1,1
8,Carrite,BDD,1,1
9,Someguy1221,BDD,1,1


In [83]:
df.shape

(198275, 4)

In [84]:
df.dtypes

SOURCE    object
TARGET    object
VOTE       int64
RESULT     int64
dtype: object

In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 198275 entries, 0 to 198274
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   SOURCE  196614 non-null  object
 1   TARGET  198275 non-null  object
 2   VOTE    198275 non-null  int64 
 3   RESULT  198275 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 7.6+ MB


In [86]:
resultado = df["TARGET"].nunique()
contagem =df["TARGET"].value_counts()
print("Número de classes únicas na coluna TARGET:", resultado)
print("Contagem de cada classe na coluna TARGET:")
print(contagem)
# mostra os nós que receberam mais e menos votos em target

Número de classes únicas na coluna TARGET: 3497
Contagem de cada classe na coluna TARGET:
TARGET
Everyking         987
TenPoundHammer    778
MZMcBride         663
Cobi              639
Elonka            628
                 ... 
Jdforrester         1
Patrick             1
Olivier             1
Mintguy             1
Zanimum             1
Name: count, Length: 3497, dtype: int64


In [87]:
df.isna().sum()

SOURCE    1661
TARGET       0
VOTE         0
RESULT       0
dtype: int64

In [88]:
df[df['SOURCE'].isna()]

,SOURCE,TARGET,VOTE,RESULT
707,NaN,Jason Quinn,0,1
708,NaN,Jason Quinn,0,1
793,NaN,Legoktm,1,1
1126,NaN,Mattythewhite,-1,1
1128,NaN,Mattythewhite,-1,1
...,...,...,...,...
198052,NaN,Yelyos,1,-1
198116,NaN,Christopher Mahan,1,1
198171,NaN,Jwrosenzweig,1,1
198217,NaN,Patrick,1,1


In [89]:
#remove votos sem identificação de quem votou
antes = len(df)

df = df.dropna(subset=["SOURCE", "TARGET"]).copy()

print("Antes:", antes)
print("Depois:", len(df))
print("Removidos:", antes - len(df))

Antes: 198275
Depois: 196614
Removidos: 1661


In [90]:
print(df["VOTE"].value_counts().sort_index())
print(df["RESULT"].value_counts().sort_index())

VOTE
-1     40667
 0     12068
 1    143879
Name: count, dtype: int64
RESULT
-1     74090
 1    122524
Name: count, dtype: int64


In [91]:
resultados = df.groupby("TARGET")["RESULT"].nunique()

print("Candidatos com resultados diferentes:", (resultados > 1).sum())

Candidatos com resultados diferentes: 236
